## Decision Tree for Time Series Forecasting
#### Enhanced Decision Tree with Ensemble Methods for Restaurant Demand Prediction
- **AVAILABLE** for multivariate time-series with interpretable decision rules
- Excellent handling of non-linear relationships and feature interactions
- Built-in feature selection and robust to outliers
- Easy interpretation and business rule extraction


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [2]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']


# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []


In [3]:
# Comprehensive Feature Engineering for Decision Trees

def create_time_features(df):
    """Create time-based features optimized for Decision Trees"""
    df = df.copy()
    
    # Basic time features
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    
    # Cyclical encoding for Decision Trees (binned approach)
    df['month_sin_binned'] = pd.cut(np.sin(2 * np.pi * df['month'] / 12), bins=10, labels=False)
    df['month_cos_binned'] = pd.cut(np.cos(2 * np.pi * df['month'] / 12), bins=10, labels=False)
    df['dayofweek_sin_binned'] = pd.cut(np.sin(2 * np.pi * df['dayofweek'] / 7), bins=7, labels=False)
    df['dayofweek_cos_binned'] = pd.cut(np.cos(2 * np.pi * df['dayofweek'] / 7), bins=7, labels=False)
    
    # Holiday and special day features
    df['is_holiday'] = 0  # Placeholder - can be enhanced with actual holiday data
    df['days_since_holiday'] = 0  # Placeholder
    df['days_until_holiday'] = 0  # Placeholder
    
    # Business specific time features
    df['season'] = df['month'].apply(lambda x: 
        0 if x in [12,1,2] else  # winter
        1 if x in [3,4,5] else   # spring
        2 if x in [6,7,8] else   # summer
        3)                       # fall
    
    df['meal_time'] = df['dayofweek'].apply(lambda x:
        0 if x < 5 else          # weekday
        1)                       # weekend
    
    # Time since epoch features
    df['days_from_start'] = (df['date'] - df['date'].min()).dt.days
    df['weeks_from_start'] = df['days_from_start'] // 7
    df['months_from_start'] = ((df['date'].dt.year - df['date'].min().year) * 12 + 
                              df['date'].dt.month - df['date'].min().month)
    
    return df

def create_lag_features(df, target_col='sales', lags=[1, 2, 3, 7, 14, 21, 28]):
    """Create lag features optimized for Decision Trees"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for lag in lags:
        # Basic lag
        df[f'{target_col}_lag_{lag}'] = df.groupby('store_menu_id')[target_col].shift(lag)
        
        # Lag differences and ratios
        if lag > 1:
            df[f'{target_col}_lag_diff_{lag}'] = (
                df.groupby('store_menu_id')[target_col].shift(lag) - 
                df.groupby('store_menu_id')[target_col].shift(lag*2)
            )
            
            # Binned ratios (better for decision trees)
            ratio = (df.groupby('store_menu_id')[target_col].shift(lag) / 
                    (df.groupby('store_menu_id')[target_col].shift(lag*2) + 0.001))
            df[f'{target_col}_lag_ratio_binned_{lag}'] = pd.cut(ratio, bins=10, labels=False)
        
        # Lag trend indicators
        if lag >= 7:
            trend = (df.groupby('store_menu_id')[target_col].shift(lag) > 
                    df.groupby('store_menu_id')[target_col].shift(lag+7)).astype(int)
            df[f'{target_col}_trend_up_{lag}'] = trend
    
    return df

def create_rolling_features(df, target_col='sales', windows=[3, 7, 14, 28]):
    """Create rolling window features for Decision Trees"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for window in windows:
        # Rolling statistics
        group_rolling = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1))
        
        df[f'{target_col}_rolling_mean_{window}'] = group_rolling.mean()
        df[f'{target_col}_rolling_std_{window}'] = group_rolling.std().fillna(0)
        df[f'{target_col}_rolling_min_{window}'] = group_rolling.min()
        df[f'{target_col}_rolling_max_{window}'] = group_rolling.max()
        df[f'{target_col}_rolling_median_{window}'] = group_rolling.median()
        
        # Rolling binned features (better for trees)
        rolling_mean = group_rolling.mean()
        df[f'{target_col}_rolling_mean_binned_{window}'] = pd.cut(rolling_mean, bins=10, labels=False)
        
        # Rolling volatility indicators
        rolling_std = group_rolling.std().fillna(0)
        df[f'{target_col}_high_volatility_{window}'] = (rolling_std > rolling_std.quantile(0.8)).astype(int)
        df[f'{target_col}_low_volatility_{window}'] = (rolling_std < rolling_std.quantile(0.2)).astype(int)
        
        # Rolling momentum indicators
        current_vs_mean = df[target_col] - rolling_mean
        df[f'{target_col}_above_rolling_mean_{window}'] = (current_vs_mean > 0).astype(int)
        df[f'{target_col}_momentum_strength_{window}'] = pd.cut(
            abs(current_vs_mean), bins=5, labels=False
        )
        
        # Range-based features
        rolling_range = group_rolling.max() - group_rolling.min()
        df[f'{target_col}_rolling_range_{window}'] = rolling_range
        df[f'{target_col}_rolling_range_binned_{window}'] = pd.cut(rolling_range, bins=5, labels=False)
    
    return df

def create_interaction_features(df):
    """Create interaction features for Decision Trees"""
    df = df.copy()
    
    # Encode categorical variables for interactions
    le_store = LabelEncoder()
    le_menu = LabelEncoder()
    
    df['store_encoded'] = le_store.fit_transform(df['store'])
    df['menu_encoded'] = le_menu.fit_transform(df['menu'])
    
    # Store-time interactions
    df['store_dayofweek'] = df['store_encoded'] * 10 + df['dayofweek']
    df['store_month'] = df['store_encoded'] * 100 + df['month']
    df['store_season'] = df['store_encoded'] * 10 + df['season']
    df['store_weekend'] = df['store_encoded'] * 10 + df['is_weekend']
    
    # Menu-time interactions
    df['menu_dayofweek'] = df['menu_encoded'] * 10 + df['dayofweek']
    df['menu_month'] = df['menu_encoded'] * 100 + df['month']
    df['menu_season'] = df['menu_encoded'] * 10 + df['season']
    
    # Complex interactions
    df['store_menu_interaction'] = df['store_encoded'] * 1000 + df['menu_encoded']
    df['store_menu_dayofweek'] = df['store_menu_interaction'] * 10 + df['dayofweek']
    df['store_menu_month'] = df['store_menu_interaction'] * 100 + df['month']
    
    # Business logic interactions
    df['weekend_season'] = df['is_weekend'] * 10 + df['season']
    df['month_dayofweek'] = df['month'] * 10 + df['dayofweek']
    
    return df

def create_target_statistics(df, categorical_cols, target_col='sales', min_samples=5):
    """Create target statistics for categorical variables"""
    df = df.copy()
    
    for col in categorical_cols:
        if col in df.columns:
            # Basic target statistics
            target_stats = df.groupby(col)[target_col].agg(['mean', 'std', 'count', 'median'])
            target_stats = target_stats.fillna(df[target_col].mean())
            
            # Only use statistics for categories with sufficient samples
            target_stats.loc[target_stats['count'] < min_samples, ['mean', 'std', 'median']] = df[target_col].mean()
            
            df[f'{col}_target_mean'] = df[col].map(target_stats['mean'])
            df[f'{col}_target_std'] = df[col].map(target_stats['std']).fillna(0)
            df[f'{col}_target_median'] = df[col].map(target_stats['median'])
            df[f'{col}_target_count'] = df[col].map(target_stats['count'])
            
            # Binned target statistics (better for trees)
            df[f'{col}_target_mean_binned'] = pd.cut(df[f'{col}_target_mean'], bins=10, labels=False)
            df[f'{col}_high_volume_category'] = (df[f'{col}_target_count'] > target_stats['count'].quantile(0.8)).astype(int)
            df[f'{col}_low_volume_category'] = (df[f'{col}_target_count'] < target_stats['count'].quantile(0.2)).astype(int)
    
    return df


In [4]:
def prepare_features_for_trees(df):
    """Prepare all features optimized for Decision Trees"""
    df = df.copy()
    
    # Time features
    df = create_time_features(df)
    
    # Interaction features
    df = create_interaction_features(df)
    
    # Lag features
    df = create_lag_features(df)
    
    # Rolling features
    df = create_rolling_features(df)
    
    # Target statistics
    categorical_cols = ['store_encoded', 'menu_encoded', 'store_dayofweek', 'menu_dayofweek']
    df = create_target_statistics(df, categorical_cols)
    
    return df

def get_feature_columns_for_trees(df):
    """Get feature columns for Decision Tree models"""
    exclude_cols = [
        'date', 'store_menu_id', 'sales', 'date_ordinal', 'store', 'menu'
    ]
    
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    return feature_cols

def select_best_features(X, y, n_features=50, method='mutual_info'):
    """Select best features using various methods"""
    
    if method == 'f_regression':
        selector = SelectKBest(score_func=f_regression, k=min(n_features, X.shape[1]))
        X_selected = selector.fit_transform(X, y)
        selected_features = X.columns[selector.get_support()].tolist()
        
    elif method == 'tree_importance':
        # Use a simple tree to get feature importance
        tree = ExtraTreeRegressor(random_state=42)
        tree.fit(X.fillna(0), y)
        
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': tree.feature_importances_
        }).sort_values('importance', ascending=False)
        
        selected_features = feature_importance.head(n_features)['feature'].tolist()
        X_selected = X[selected_features]
        
    else:  # default to all features with basic filtering
        # Remove features with too many missing values
        missing_pct = X.isnull().sum() / len(X)
        good_features = missing_pct[missing_pct < 0.5].index.tolist()
        selected_features = good_features[:n_features] if len(good_features) > n_features else good_features
        X_selected = X[selected_features]
    
    return X_selected, selected_features


In [5]:
def get_tree_models():
    """Get various Decision Tree models with optimized parameters"""
    
    models = {
        'decision_tree': DecisionTreeRegressor(
            max_depth=15,
            min_samples_split=20,
            min_samples_leaf=10,
            max_features='sqrt',
            random_state=42,
            ccp_alpha=0.001  # Cost complexity pruning
        ),
        
        'extra_tree': ExtraTreeRegressor(
            max_depth=20,
            min_samples_split=15,
            min_samples_leaf=8,
            max_features='sqrt',
            random_state=42
        ),
        
        'random_forest': RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            min_samples_split=25,
            min_samples_leaf=12,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1,
            bootstrap=True,
            oob_score=True
        ),
        
        'extra_trees': ExtraTreesRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=20,
            min_samples_leaf=10,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1,
            bootstrap=False
        ),
        
        'gradient_boosting': GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=8,
            min_samples_split=20,
            min_samples_leaf=15,
            max_features='sqrt',
            random_state=42,
            subsample=0.8
        )
    }
    
    return models

def predict_with_decision_trees(train_df, test_df, sid, model_type='random_forest'):
    """Predict using Decision Tree models for a specific store_menu_id"""
    
    # Filter data for specific store_menu_id
    train_sid = train_df[train_df['store_menu_id'] == sid].copy()
    test_sid = test_df[test_df['store_menu_id'] == sid].copy()
    
    if len(train_sid) == 0 or len(test_sid) == 0:
        raise ValueError(f"No data found for {sid}")
    
    # Combine and sort data
    combined_data = pd.concat([train_sid, test_sid], ignore_index=True)
    combined_data = combined_data.sort_values('date').reset_index(drop=True)
    
    # Prepare features
    combined_data = prepare_features_for_trees(combined_data)
    
    # Get feature columns
    feature_cols = get_feature_columns_for_trees(combined_data)
    
    # Split back to train and test
    train_end_idx = len(train_sid)
    train_features = combined_data.iloc[:train_end_idx]
    test_features = combined_data.iloc[train_end_idx:]
    
    # Get last 28 days for prediction input
    input_end_ordinal = test_features['date_ordinal'].max()
    prediction_input = combined_data[
        (combined_data['date_ordinal'] <= input_end_ordinal) & 
        (combined_data['date_ordinal'] > input_end_ordinal - 28)
    ].copy()
    
    if len(prediction_input) != 28:
        raise ValueError(f"{sid} does not have exactly 28 days of input data.")
    
    # Prepare training data
    train_data_for_model = combined_data[
        combined_data['date_ordinal'] <= input_end_ordinal
    ].copy()
    
    # Remove rows with missing target
    train_data_for_model = train_data_for_model.dropna(subset=['sales'])
    
    if len(train_data_for_model) < 50:  # Minimum training samples for trees
        # Fallback to simple mean prediction
        recent_sales = prediction_input['sales'].dropna()
        recent_mean = recent_sales.tail(14).mean() if len(recent_sales) > 0 else 0
        forecast = np.full(7, recent_mean if not pd.isna(recent_mean) else 0)
    else:
        # Prepare features and target
        X_train = train_data_for_model[feature_cols].fillna(0)
        y_train = train_data_for_model['sales']
        
        # Feature selection for better performance
        if len(feature_cols) > 100:
            X_train_selected, selected_features = select_best_features(
                X_train, y_train, n_features=100, method='tree_importance'
            )
        else:
            X_train_selected = X_train
            selected_features = feature_cols
        
        # Get the appropriate model
        models = get_tree_models()
        model = models.get(model_type, models['random_forest'])
        
        # Train model
        model.fit(X_train_selected, y_train)
        
        # Predict next 7 days iteratively
        forecast = []
        current_data = combined_data.copy()
        
        for day in range(7):
            # Create next day data
            next_date = prediction_input['date'].max() + timedelta(days=day+1)
            next_ordinal = input_end_ordinal + day + 1
            
            # Create a new row for prediction
            next_row = prediction_input.iloc[-1:].copy()
            next_row['date'] = next_date
            next_row['date_ordinal'] = next_ordinal
            next_row['sales'] = None  # Unknown target
            
            # Add to current data and recreate features
            temp_data = pd.concat([current_data, next_row], ignore_index=True)
            temp_data = temp_data.sort_values('date').reset_index(drop=True)
            temp_data = prepare_features_for_trees(temp_data)
            
            # Get the prediction row
            pred_row = temp_data.iloc[-1:]
            
            # Handle missing values and select features
            pred_features = pred_row[selected_features].fillna(0)
            
            # Make prediction
            pred_value = model.predict(pred_features)[0]
            pred_value = max(0, pred_value)  # Ensure non-negative
            
            forecast.append(pred_value)
            
            # Update the prediction row with the predicted value
            temp_data.loc[temp_data.index[-1], 'sales'] = pred_value
            current_data = temp_data.copy()
        
        forecast = np.array(forecast)
    
    # Create forecast dates
    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
    
    return pd.DataFrame({
        'date': forecast_dates,
        'store_menu_id': sid,
        'sales': forecast
    })


In [6]:
def predict_with_ensemble_trees(train_df, test_df, sid, ensemble_methods=['random_forest', 'extra_trees', 'gradient_boosting']):
    """Predict using ensemble of Decision Tree models"""
    
    predictions = []
    weights = []
    
    for method in ensemble_methods:
        try:
            pred_df = predict_with_decision_trees(train_df, test_df, sid, model_type=method)
            predictions.append(pred_df['sales'].values)
            
            # Weight based on model type (can be optimized based on validation)
            if method == 'random_forest':
                weights.append(0.4)
            elif method == 'extra_trees':
                weights.append(0.3)
            elif method == 'gradient_boosting':
                weights.append(0.3)
            else:
                weights.append(0.2)
                
        except Exception as e:
            print(f"Method {method} failed for {sid}: {e}")
            continue
    
    if not predictions:
        raise ValueError(f"All ensemble methods failed for {sid}")
    
    # Normalize weights
    weights = np.array(weights[:len(predictions)])
    weights = weights / weights.sum()
    
    # Weighted ensemble prediction
    ensemble_forecast = np.average(predictions, axis=0, weights=weights)
    
    # Get the base prediction structure
    base_pred = predict_with_decision_trees(train_df, test_df, sid, model_type=ensemble_methods[0])
    base_pred['sales'] = ensemble_forecast
    
    return base_pred

def run_recursive_forecasting_trees(train_df, test_data_list, use_ensemble=True):
    """Run recursive forecasting using Decision Tree models"""
    all_predictions = []
    
    for i, test_df in enumerate(test_data_list):
        test_df = test_df.copy()
        test_df['date'] = pd.to_datetime(test_df['date'])
        test_df = test_df.sort_values(['store_menu_id', 'date'])
        
        pred_list = []
        store_menu_ids = test_df['store_menu_id'].unique()
        
        for sid in tqdm(store_menu_ids, desc=f"Predicting TEST_{i} with Decision Trees"):
            try:
                if use_ensemble:
                    pred_df = predict_with_ensemble_trees(train_df, test_df, sid)
                else:
                    pred_df = predict_with_decision_trees(train_df, test_df, sid, model_type='random_forest')
                
                pred_list.append(pred_df)
                
                # Update train_df: add current test + prediction
                test_part = test_df[test_df['store_menu_id'] == sid]
                train_df = pd.concat([train_df, test_part, pred_df])
                
            except Exception as e:
                print(f"Failed for {sid}: {e}")
                # Create fallback prediction
                test_part = test_df[test_df['store_menu_id'] == sid]
                if len(test_part) > 0:
                    input_end_ordinal = test_part['date_ordinal'].max()
                    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
                    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
                    
                    # Use recent median as fallback (more robust than mean)
                    recent_median = test_part['sales'].tail(14).median()
                    fallback_value = recent_median if not pd.isna(recent_median) else 0
                    
                    fallback_pred = pd.DataFrame({
                        'date': forecast_dates,
                        'store_menu_id': sid,
                        'sales': np.full(7, fallback_value)
                    })
                    pred_list.append(fallback_pred)
                    
                    # Update train_df with test data and fallback prediction
                    train_df = pd.concat([train_df, test_part, fallback_pred])
        
        if pred_list:
            all_predictions.append(pd.concat(pred_list))
    
    return pd.concat(all_predictions) if all_predictions else pd.DataFrame()


In [7]:
# Prepare test data list
test_data_list = []
for i in range(10):
    test_data_list.append(globals()[f'test_data_{i}'])

# Run Decision Tree forecasting
print("Starting Decision Tree recursive forecasting...")
print("Using ensemble of Random Forest, Extra Trees, and Gradient Boosting...")

final_predictions = run_recursive_forecasting_trees(
    train_data.copy(), 
    test_data_list, 
    use_ensemble=True
)

print(f"Total predictions generated: {len(final_predictions)}")
if len(final_predictions) > 0:
    print("First few predictions:")
    print(final_predictions.head(10))


Starting Decision Tree recursive forecasting...
Using ensemble of Random Forest, Extra Trees, and Gradient Boosting...


Predicting TEST_0 with Decision Trees:   0%|          | 0/193 [00:00<?, ?it/s]

Method random_forest failed for 느티나무 셀프BBQ_1인 수저세트: cannot convert the series to <class 'float'>
Method extra_trees failed for 느티나무 셀프BBQ_1인 수저세트: cannot convert the series to <class 'float'>


Predicting TEST_0 with Decision Trees:   1%|          | 1/193 [00:00<01:16,  2.53it/s]

Method gradient_boosting failed for 느티나무 셀프BBQ_1인 수저세트: cannot convert the series to <class 'float'>
Failed for 느티나무 셀프BBQ_1인 수저세트: All ensemble methods failed for 느티나무 셀프BBQ_1인 수저세트
Method random_forest failed for 느티나무 셀프BBQ_BBQ55(단체): cannot convert the series to <class 'float'>
Method extra_trees failed for 느티나무 셀프BBQ_BBQ55(단체): cannot convert the series to <class 'float'>


...

Method extra_trees failed for 화담숲카페_아메리카노 ICE: cannot convert the series to <class 'float'>
Method gradient_boosting failed for 화담숲카페_아메리카노 ICE: cannot convert the series to <class 'float'>
Failed for 화담숲카페_아메리카노 ICE: All ensemble methods failed for 화담숲카페_아메리카노 ICE
Method random_forest failed for 화담숲카페_카페라떼 ICE: cannot convert the series to <class 'float'>
Method extra_trees failed for 화담숲카페_카페라떼 ICE: cannot convert the series to <class 'float'>


Predicting TEST_9 with Decision Trees:  99%|█████████▉| 192/193 [01:52<00:00,  1.70it/s]

Method gradient_boosting failed for 화담숲카페_카페라떼 ICE: cannot convert the series to <class 'float'>
Failed for 화담숲카페_카페라떼 ICE: All ensemble methods failed for 화담숲카페_카페라떼 ICE
Method random_forest failed for 화담숲카페_현미뻥스크림: cannot convert the series to <class 'float'>
Method extra_trees failed for 화담숲카페_현미뻥스크림: cannot convert the series to <class 'float'>


Predicting TEST_9 with Decision Trees: 100%|██████████| 193/193 [01:52<00:00,  1.71it/s]

Method gradient_boosting failed for 화담숲카페_현미뻥스크림: cannot convert the series to <class 'float'>
Failed for 화담숲카페_현미뻥스크림: All ensemble methods failed for 화담숲카페_현미뻥스크림
Total predictions generated: 13510
First few predictions:
        date         store_menu_id  sales
0 2024-07-14    느티나무 셀프BBQ_1인 수저세트    4.5
1 2024-07-15    느티나무 셀프BBQ_1인 수저세트    4.5
2 2024-07-16    느티나무 셀프BBQ_1인 수저세트    4.5
3 2024-07-17    느티나무 셀프BBQ_1인 수저세트    4.5
4 2024-07-18    느티나무 셀프BBQ_1인 수저세트    4.5
5 2024-07-19    느티나무 셀프BBQ_1인 수저세트    4.5
6 2024-07-20    느티나무 셀프BBQ_1인 수저세트    4.5
0 2024-07-14  느티나무 셀프BBQ_BBQ55(단체)    0.0
1 2024-07-15  느티나무 셀프BBQ_BBQ55(단체)    0.0
2 2024-07-16  느티나무 셀프BBQ_BBQ55(단체)    0.0


In [8]:
# Model evaluation and performance analysis
def evaluate_tree_performance(predictions_df):
    """Comprehensive evaluation of Decision Tree model performance"""
    
    if len(predictions_df) == 0:
        print("No predictions to evaluate.")
        return
        
    print("=== Decision Tree Model Performance Summary ===")
    print(f"Total predictions: {len(predictions_df)}")
    print(f"Unique store-menu combinations: {predictions_df['store_menu_id'].nunique()}")
    print(f"Prediction date range: {predictions_df['date'].min()} to {predictions_df['date'].max()}")
    
    print("\n=== Sales Prediction Statistics ===")
    print(predictions_df['sales'].describe())
    
    # Check for negative predictions
    negative_count = (predictions_df['sales'] < 0).sum()
    print(f"\nNegative predictions: {negative_count}")
    
    # Check for missing values
    missing_count = predictions_df['sales'].isna().sum()
    print(f"Missing predictions: {missing_count}")
    
    print("\n=== Top 10 Store-Menu by Predicted Sales ===")
    top_predictions = predictions_df.groupby('store_menu_id')['sales'].sum().sort_values(ascending=False).head(10)
    for idx, (store_menu, total_sales) in enumerate(top_predictions.items(), 1):
        print(f"{idx:2d}. {store_menu}: {total_sales:.2f}")
    
    print("\n=== Daily Prediction Patterns ===")
    daily_stats = predictions_df.groupby('date')['sales'].agg(['count', 'mean', 'std', 'min', 'max']).round(2)
    print(daily_stats)
    
    print("\n=== Store-wise Prediction Summary ===")
    if 'store' in predictions_df['store_menu_id'].str.split('_').str[0].values:
        predictions_df_temp = predictions_df.copy()
        predictions_df_temp['store'] = predictions_df_temp['store_menu_id'].str.split('_').str[0]
        store_stats = predictions_df_temp.groupby('store')['sales'].agg(['count', 'mean', 'sum']).round(2)
        print(store_stats)
    
    # Prediction distribution analysis
    print("\n=== Prediction Distribution Analysis ===")
    zero_predictions = (predictions_df['sales'] == 0).sum()
    low_predictions = ((predictions_df['sales'] > 0) & (predictions_df['sales'] <= 5)).sum()
    medium_predictions = ((predictions_df['sales'] > 5) & (predictions_df['sales'] <= 20)).sum()
    high_predictions = (predictions_df['sales'] > 20).sum()
    
    print(f"Zero predictions: {zero_predictions} ({zero_predictions/len(predictions_df)*100:.1f}%)")
    print(f"Low predictions (0-5): {low_predictions} ({low_predictions/len(predictions_df)*100:.1f}%)")
    print(f"Medium predictions (5-20): {medium_predictions} ({medium_predictions/len(predictions_df)*100:.1f}%)")
    print(f"High predictions (>20): {high_predictions} ({high_predictions/len(predictions_df)*100:.1f}%)")

if len(final_predictions) > 0:
    evaluate_tree_performance(final_predictions)


=== Decision Tree Model Performance Summary ===
Total predictions: 13510
Unique store-menu combinations: 193
Prediction date range: 2024-07-14 00:00:00 to 2025-05-31 00:00:00

=== Sales Prediction Statistics ===
count    13510.000000
mean         6.963212
std         27.831279
min          0.000000
25%          0.000000
50%          0.500000
75%          3.000000
max        525.000000
Name: sales, dtype: float64

Negative predictions: 0
Missing predictions: 0

=== Top 10 Store-Menu by Predicted Sales ===
 1. 화담숲주막_해물파전: 7812.00
 2. 포레스트릿_꼬치어묵: 5880.00
 3. 포레스트릿_떡볶이: 3563.00
 4. 카페테리아_수제 등심 돈까스: 3258.50
 5. 포레스트릿_생수: 2982.00
 6. 화담숲카페_아메리카노 ICE: 2737.00
 7. 포레스트릿_치즈 핫도그: 2509.50
 8. 미라시아_브런치(대인) 주중: 2418.50
 9. 카페테리아_단체식 18000(신): 2026.50
10. 화담숲카페_현미뻥스크림: 2002.00

=== Daily Prediction Patterns ===
            count  mean    std  min   max
date                                     
2024-07-14    193  1.87   4.68  0.0  40.0
2024-07-15    193  1.87   4.68  0.0  40.0
2024-07-16    193  1.87

In [9]:
# Feature importance analysis for Decision Trees
def analyze_feature_importance_trees(sample_data, model_type='random_forest'):
    """Analyze feature importance using Decision Tree models"""
    
    print(f"=== Decision Tree Feature Importance Analysis ({model_type}) ===")
    print("Training a sample model to analyze feature importance...")
    
    try:
        # Prepare sample data
        sample_prepared = prepare_features_for_trees(sample_data)
        feature_cols = get_feature_columns_for_trees(sample_prepared)
        
        # Remove rows with missing target values
        sample_clean = sample_prepared.dropna(subset=['sales'])
        
        if len(sample_clean) < 200:
            print("Not enough clean data for feature importance analysis.")
            return
        
        # Prepare features and target
        X = sample_clean[feature_cols].fillna(0)
        y = sample_clean['sales']
        
        # Simple train-test split
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
        
        # Feature selection
        if len(feature_cols) > 100:
            X_train_selected, selected_features = select_best_features(
                X_train, y_train, n_features=100, method='tree_importance'
            )
            X_val_selected = X_val[selected_features]
        else:
            X_train_selected = X_train
            X_val_selected = X_val
            selected_features = feature_cols
        
        # Train the model
        models = get_tree_models()
        model = models.get(model_type, models['random_forest'])
        
        model.fit(X_train_selected, y_train)
        
        # Get feature importance
        if hasattr(model, 'feature_importances_'):
            feature_importance = pd.DataFrame({
                'feature': selected_features,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)
            
            print("\nTop 20 Most Important Features:")
            print(feature_importance.head(20).to_string(index=False))
            
            # Analyze feature types
            print("\nFeature Importance by Type:")
            
            # Categorize features
            lag_features = feature_importance[feature_importance['feature'].str.contains('lag')]
            rolling_features = feature_importance[feature_importance['feature'].str.contains('rolling')]
            time_features = feature_importance[feature_importance['feature'].str.contains('month|day|year|season|weekend')]
            interaction_features = feature_importance[feature_importance['feature'].str.contains('_')]
            
            print(f"Top Lag Features:")
            print(lag_features.head(5).to_string(index=False))
            
            print(f"\nTop Rolling Features:")
            print(rolling_features.head(5).to_string(index=False))
            
            print(f"\nTop Time Features:")
            print(time_features.head(5).to_string(index=False))
        
        # Validation performance
        y_pred = model.predict(X_val_selected)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        
        print(f"\nSample Model Performance:")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R²: {r2:.4f}")
        
        # Model specific information
        if hasattr(model, 'n_estimators'):
            print(f"\nModel Info:")
            print(f"Number of estimators: {model.n_estimators}")
            if hasattr(model, 'oob_score_'):
                print(f"OOB Score: {model.oob_score_:.4f}")
        
        return feature_importance
        
    except Exception as e:
        print(f"Feature importance analysis failed: {e}")
        return None

# Run feature importance analysis on a sample
if len(train_data) > 1000:
    sample_size = min(5000, len(train_data))
    sample_data = train_data.sample(n=sample_size, random_state=42)
    
    # Analyze different models
    for model_type in ['random_forest', 'extra_trees', 'gradient_boosting']:
        importance_df = analyze_feature_importance_trees(sample_data, model_type)
        print("\n" + "="*80 + "\n")


=== Decision Tree Feature Importance Analysis (random_forest) ===
Training a sample model to analyze feature importance...
Feature importance analysis failed: Bin edges must be unique: Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg


=== Decision Tree Feature Importance Analysis (extra_trees) ===
Training a sample model to analyze feature importance...
Feature importance analysis failed: Bin edges must be unique: Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg


=== Decision Tree Feature Importance Analysis (gradient_boosting) ===
Training a sample model to analyze feature importance...
Feature importance analysis failed: Bin edges must be unique: Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg




In [10]:
# Create submission file
def create_submission_file(predictions_df, submission_template, output_path):
    """Create submission file in the required format"""
    
    if len(predictions_df) == 0:
        print("No predictions to create submission file.")
        return None
    
    # Initialize submission with template
    submission_df = submission_template.copy()
    
    # Group predictions by store_menu_id and date
    predictions_pivot = predictions_df.pivot_table(
        index='date', 
        columns='store_menu_id', 
        values='sales', 
        fill_value=0
    )
    
    print(f"Predictions pivot shape: {predictions_pivot.shape}")
    print(f"Submission template shape: {submission_df.shape}")
    
    # Fill submission template with predictions
    for col in submission_df.columns:
        if col in predictions_pivot.columns:
            # Get predictions for this store_menu combination
            pred_values = predictions_pivot[col].values
            if len(pred_values) == len(submission_df):
                submission_df[col] = pred_values
            else:
                print(f"Warning: Mismatch in prediction length for {col}: {len(pred_values)} vs {len(submission_df)}")
                # Fill with available predictions or zeros
                min_len = min(len(pred_values), len(submission_df))
                submission_df.loc[:min_len-1, col] = pred_values[:min_len]
                if min_len < len(submission_df):
                    submission_df.loc[min_len:, col] = 0
        else:
            print(f"Warning: No predictions found for {col}")
            submission_df[col] = 0  # Fill with zeros if no prediction
    
    # Ensure no negative values
    numeric_cols = submission_df.select_dtypes(include=[np.number]).columns
    submission_df[numeric_cols] = submission_df[numeric_cols].clip(lower=0)
    
    # Save submission file
    submission_df.to_csv(output_path, index=False)
    print(f"Submission file saved to: {output_path}")
    
    return submission_df

# Create and save submission
if len(final_predictions) > 0:
    submission_result = create_submission_file(
        final_predictions, 
        submission, 
        "./result/decision_tree_ensemble_submission.csv"
    )
    
    if submission_result is not None:
        print("\nSubmission file shape:", submission_result.shape)
        print("Sample submission values:")
        print(submission_result.iloc[:5, :5])
        
        # Check for missing values
        missing_count = submission_result.isna().sum().sum()
        print(f"\nTotal missing values in submission: {missing_count}")
        
        # Basic statistics
        numeric_cols = submission_result.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print("\nSubmission statistics:")
            print(f"Mean: {submission_result[numeric_cols].mean().mean():.4f}")
            print(f"Std: {submission_result[numeric_cols].std().mean():.4f}")
            print(f"Min: {submission_result[numeric_cols].min().min():.4f}")
            print(f"Max: {submission_result[numeric_cols].max().max():.4f}")
            print(f"Negative values: {(submission_result[numeric_cols] < 0).sum().sum()}")
else:
    print("No predictions generated. Please check the model.")


Predictions pivot shape: (70, 193)
Submission template shape: (70, 194)
Submission file saved to: ./result/decision_tree_ensemble_submission.csv

Submission file shape: (70, 194)
Sample submission values:
   date  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  느티나무 셀프BBQ_대여료 30,000원  \
0     0                 4.5                   0.0                     2.0   
1     0                 4.5                   0.0                     2.0   
2     0                 4.5                   0.0                     2.0   
3     0                 4.5                   0.0                     2.0   
4     0                 4.5                   0.0                     2.0   

   느티나무 셀프BBQ_대여료 60,000원  
0                     1.5  
1                     1.5  
2                     1.5  
3                     1.5  
4                     1.5  

Total missing values in submission: 0

Submission statistics:
Mean: 6.9273
Std: 10.3461
Min: 0.0000
Max: 525.0000
Negative values: 0


In [11]:
# Decision Tree advantages for this use case
def decision_tree_advantages():
    """Display Decision Tree advantages for restaurant demand forecasting"""
    
    print("=== Decision Tree Advantages for Restaurant Demand Forecasting ===")
    print()
    print("1. **Interpretability and Business Rules:**")
    print("   - Easy to understand decision paths")
    print("   - Can extract business rules (e.g., 'If weekend and season=summer, then sales > X')")
    print("   - Stakeholders can easily validate model logic")
    print()
    print("2. **Non-linear Relationship Handling:**")
    print("   - Captures complex interactions between features")
    print("   - No assumptions about data distribution")
    print("   - Handles threshold effects (e.g., different behavior above/below certain sales levels)")
    print()
    print("3. **Robust to Outliers and Missing Data:**")
    print("   - Tree splits naturally handle outliers")
    print("   - Can handle missing values through surrogate splits")
    print("   - No need for extensive data preprocessing")
    print()
    print("4. **Feature Selection and Interaction Detection:**")
    print("   - Automatic feature selection through splits")
    print("   - Identifies important feature interactions")
    print("   - Reduces overfitting through pruning")
    print()
    print("5. **Ensemble Methods Benefits:**")
    print("   - Random Forest: Reduces overfitting, handles high-dimensional data")
    print("   - Extra Trees: Faster training, additional randomness")
    print("   - Gradient Boosting: Sequential learning, high accuracy")
    print()
    print("6. **Business Context Specific:**")
    print("   - Handles seasonal patterns through explicit rules")
    print("   - Captures store-specific behaviors")
    print("   - Identifies menu item interactions")
    print("   - Works well with categorical features (store, menu types)")
    print()
    print("7. **Practical Advantages:**")
    print("   - Fast prediction after training")
    print("   - Scales well with data size")
    print("   - No hyperparameter tuning required for basic models")
    print("   - Memory efficient")
    print()

decision_tree_advantages()


=== Decision Tree Advantages for Restaurant Demand Forecasting ===

1. **Interpretability and Business Rules:**
   - Easy to understand decision paths
   - Can extract business rules (e.g., 'If weekend and season=summer, then sales > X')
   - Stakeholders can easily validate model logic

2. **Non-linear Relationship Handling:**
   - Captures complex interactions between features
   - No assumptions about data distribution
   - Handles threshold effects (e.g., different behavior above/below certain sales levels)

3. **Robust to Outliers and Missing Data:**
   - Tree splits naturally handle outliers
   - Can handle missing values through surrogate splits
   - No need for extensive data preprocessing

4. **Feature Selection and Interaction Detection:**
   - Automatic feature selection through splits
   - Identifies important feature interactions
   - Reduces overfitting through pruning

5. **Ensemble Methods Benefits:**
   - Random Forest: Reduces overfitting, handles high-dimensional dat

In [12]:
# Hyperparameter tuning for Decision Trees (optional)
def tune_tree_hyperparameters(sample_data, model_type='random_forest'):
    """Simple hyperparameter tuning for Decision Tree models"""
    
    print(f"=== Decision Tree Hyperparameter Tuning ({model_type}) ===")
    
    try:
        # Prepare data
        sample_prepared = prepare_features_for_trees(sample_data)
        feature_cols = get_feature_columns_for_trees(sample_prepared)
        sample_clean = sample_prepared.dropna(subset=['sales'])
        
        if len(sample_clean) < 1000:
            print("Not enough data for hyperparameter tuning.")
            return None
        
        X = sample_clean[feature_cols].fillna(0)
        y = sample_clean['sales']
        
        # Feature selection
        if len(feature_cols) > 100:
            X, selected_features = select_best_features(
                X, y, n_features=100, method='tree_importance'
            )
        
        # Time series cross-validation
        tscv = TimeSeriesSplit(n_splits=3)
        
        # Define parameter grids
        if model_type == 'random_forest':
            param_combinations = [
                {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 20},
                {'n_estimators': 100, 'max_depth': 12, 'min_samples_split': 25},
                {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 30},
            ]
        elif model_type == 'extra_trees':
            param_combinations = [
                {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 15},
                {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 20},
                {'n_estimators': 150, 'max_depth': 18, 'min_samples_split': 25},
            ]
        else:  # gradient_boosting
            param_combinations = [
                {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.1},
                {'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.1},
                {'n_estimators': 150, 'max_depth': 10, 'learning_rate': 0.05},
            ]
        
        best_score = -np.inf
        best_params = None
        
        print("Testing parameter combinations...")
        
        for params in param_combinations:
            print(f"Testing params: {params}")
            
            if model_type == 'random_forest':
                model = RandomForestRegressor(
                    **params,
                    min_samples_leaf=10,
                    max_features='sqrt',
                    random_state=42,
                    n_jobs=-1
                )
            elif model_type == 'extra_trees':
                model = ExtraTreesRegressor(
                    **params,
                    min_samples_leaf=8,
                    max_features='sqrt',
                    random_state=42,
                    n_jobs=-1
                )
            else:  # gradient_boosting
                model = GradientBoostingRegressor(
                    **params,
                    min_samples_leaf=15,
                    max_features='sqrt',
                    random_state=42,
                    subsample=0.8
                )
            
            # Cross-validation
            scores = cross_val_score(
                model, X, y, cv=tscv, 
                scoring='neg_mean_squared_error', 
                n_jobs=-1
            )
            
            mean_score = scores.mean()
            print(f"CV Score (neg_MSE): {mean_score:.4f} (+/- {scores.std() * 2:.4f})")
            
            if mean_score > best_score:
                best_score = mean_score
                best_params = params
        
        print(f"\nBest parameters: {best_params}")
        print(f"Best CV score (neg_MSE): {best_score:.4f}")
        print(f"Best RMSE: {np.sqrt(-best_score):.4f}")
        
        return best_params
        
    except Exception as e:
        print(f"Hyperparameter tuning failed: {e}")
        return None

# Uncomment to run hyperparameter tuning
# if len(train_data) > 2000:
#     sample_data = train_data.sample(n=2000, random_state=42)
#     for model_type in ['random_forest', 'extra_trees']:
#         best_params = tune_tree_hyperparameters(sample_data, model_type)
#         print("\n" + "="*60 + "\n")
